# 🎓 Tutorial RAG Didático: Fases 1 a 3 (Do Zero à Observabilidade)

Bem-vindo à implementação visual e interativa do nosso Roteiro Didático!
O formato de **Jupyter Notebook** foi escolhido estrategicamente como o padrão de excelência educacional em Ciência de Dados, pois nos permite intercalar blocos densos de **explicação de conceitos (Markdown)** com a execução ativa de **código transparente (Python)**, testando-o *step-by-step*.

Neste material, implementaremos e testaremos as 3 primeiras fases da nossa especificação RAG orientada a qualidade:
1. **Preparação do Ambiente e LLMOps**.
2. **Aquisição, Exploração e Preparação dos Dados (Ingestion, Chunking & Embeddings)**.
3. **Construção do Componente Core de RAG (Retriever + Generator)**, instrumentado do zero com *Traceability* através do Langfuse.

---
## 📚 Fase 1: Preparação do Ambiente Didático

A infraestrutura contemporânea de IA generativa raramente opera num modelo isolado. Para acompanharmos esta prática, montaremos o motor de propulsão a partir destas bibliotecas líderes:
- `langchain` e `langchain-openai`: Nossos orquestradores estruturados. Eles fornecerão as interfaces de Prompts padronizadas e conexão com as enginarias da OpenAI.
- `chromadb`: O banco de dados vetorial (Vector Store) open-source, de altíssima performance estrutural, ideal para guardar nossos vetores semânticos localmente.
- `langfuse`: A espinha dorsal arquitetural de nossa Observabilidade (*Traceability*), registrando cada etapa da requisição e computando os custos para fins de baseline de performance.
- `datasets` e `pandas`: Para ingestão eficiente das provas de gabarito e modelagem gráfica estruturada.

In [2]:
# SETUP INICIAL: Descomente e execute o bloco apenas uma vez para instalar as dependências.
# %%capture
!pip install langchain langchain-openai chromadb langfuse datasets pandas deepeval ratelimit nest_asyncio

  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.7/604.7 kB 3.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 633.5/633.5 kB 25.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 28.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.4 MB/s  0:00:00
  DEPRECATION: Building 'ratelimit' using the legacy setup.py bdist_wheel mechanism, which will be

### Boas Práticas: Gestão de Segredos & Credenciais
Sistemas de produção lidam de forma rigorosa com chaves criptográficas. Embora por comodidade didática setaremos as variáveis de ambiente `os.environ` visivelmente neste notebook, a via arquitetural correta dita o uso de arquivos de cofre (como `.env` em pareamento com `python-dotenv`).

> ⚠️ **Prática Recomendada:** Crie sua conta gratuita em [cloud.langfuse.com](https://cloud.langfuse.com/) e colete suas chaves de API Públicas, Secretas e Host, além da sua clássica OpenAPI Key. Substitua-as adequadamente nos campos abaixo antes de continuar.

In [4]:
import os
from dotenv import load_dotenv

# Carrega variáveis de ambiente do arquivo .env contido na raiz do seu projeto
load_dotenv()

# Aviso didático caso o aluno esqueça de criar o arquivo .env
if not os.environ.get("OPENAI_API_KEY"):
    print("⚠️ AVISO: A variável 'OPENAI_API_KEY' não foi encontrada. Verifique seu arquivo .env!")
if not os.environ.get("LANGFUSE_PUBLIC_KEY"):
    print("⚠️ AVISO: As credenciais do Langfuse não foram encontradas. Verifique seu arquivo .env!")


---
## 🗂️ Fase 2: Aquisição e Preparação de Dados (Data Ingestion)

Não existe RAG (Recuperação e Geração) sem os "dados que deverão ser recuperados". Especialmente quando falamos de avaliar sistematicamente o RAG posteriormente na Fase 4, ter um Gabarito (Chamado de *Ground Truth* ou *Golden Dataset*) é absolutamente irrevogável. 

Nesta seção didática, validaremos esses conceitos simulando (ou fazendo requisição direta) um *dataset* de saúde referenciado, isolando em:
1. **Input**: A pergunta (o *Query*).
2. **Context**: A enciclopédia/bula base sobre a qual o conteúdo se apoia.
3. **Expected Output**: O que esperamos que a máquina extraia perfeitamente (para testarmos acuracidade futura).

In [5]:
import pandas as pd

print("⏳ Instanciando base referencial didática de perguntas/respostas...")

# Para garantir a segurança de que TODOS os alunos consigam executar este kernel sem esbarrar
# em instabilidades de rede do HuggingFace (ou permissões/arquitetura fechada do repo 'AQ-MedAI'), 
# preencheremos de imediato uma Mock List altamente controlada em formato de Dicionário
# refletindo perfeitamente a estrutura de um QA Dataset de saúde.
dados_didaticos = [
    {
        "input": "Qual a dosagem recomendada de Paracetamol para um adulto?",
        "context": "Farmacologia do Paracetamol: É primariamente empregado como antipirético e analgésico de intensidade baixa a moderada. Em pacientes adultos saudáveis, a administração recomendada compreende oscilações entre 500 mg a 1.000 mg, espaçadas em intervalos de 4 a 6 horas. Em nenhuma hipótese deve-se ultrapassar o limiar de toxicidade hepática fixado em 4.000 mg ao dia.",
        "expected_output": "Para adultos, recomenda-se 500mg a 1000mg a cada 4 a 6 horas, não ultrapassando 4000mg ao longo de um dia inteiro."
    },
    {
        "input": "Tomar Ibuprofeno e aspirina no mesmo dia pode desencadear problemas?",
        "context": "Contraindicações de AINEs: Tanto o Ibuprofeno quanto o Ácido Acetilsalicílico (Aspirina) enquadram-se na extensa classe dos Anti-inflamatórios Não Esteroidais. A ingestão sistêmica ou combinada potencializa massivamente as irritações da mucosa gástrica, aumentando progressivamente os riscos cirúrgicos de hemorragia ou surgimento de úlceras duodenais extremas.",
        "expected_output": "Sim. A combinação agrava a irritação do estômago e acentua significativamente riscos de hemorragia e ulceração."
    }
]

df_qa = pd.DataFrame(dados_didaticos)
display(df_qa)

ModuleNotFoundError: No module named 'datasets'

### 2.2 Transição Granular para a Inteligência: Chunking
A mente da máquina funciona numa arquitetura de janelas contextuais rígidas (*Context Window*). Passar PDFs inteiros de 1.000 páginas como contexto no ato da pergunta incorre no problemático viés de *Loss in the middle* (a arquitetura de Atenção dos Transformers perde foco nas informações centrais de textos grandes). 

Por isso dividimos os documentos. Um dos conceitos mais delicados é a adoção de **sobreposição narrativa (overlap)**.

> 🔎 **O Truque do Overlap:** Ao segmentar, se um parágrafo é brutalmente quebrado em ("a posologia", "indica que"), perdemos o viés semântico primário. O `chunk_overlap` contorna a quebra de sentido forçando os 40 caracteres finais do texto antigo a se recomeçarem no texto novo.

In [6]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Usaremos um fragmentador recursivo inteligente em nível de preposição, tamanho e hierarquia estrutural
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,     # Tamanho restritivo didático proposital para gerar alta multiplicidade de nós
    chunk_overlap=30    # A magia: reter parte narrativa do corte passado no atual
)

contextos_originais = df_qa['context'].tolist()

# Transformando strings puras num sistema base de Documentos abstraídos (Chunks finais)
documentos_fragmentados = text_splitter.create_documents(contextos_originais)

print(f"✅ Textos processados fisicamente. Resultando em {len(documentos_fragmentados)} fragmentos (chunks).")
print(f"\nConteúdo analítico de como o motor vê o chunk 1:")
print(f"--> '{documentos_fragmentados[1].page_content}'")

ModuleNotFoundError: No module named 'langchain'

### 2.3 Rumo à Representação Matemática (Embeddings Array)
Um sistema de busca lexical (como o saudoso *CTRL+F* do seu Word) varre as letras isoladas de uma palavra.
Um sistema Vetorial (*Vector Store*) eleva a pesquisa à compressão semântica através de dimensões numéricas. Uma ferramenta de _Embedding_ gera vetores mapeando conceitos aproximados da palavra.

Nesta etapa, persistirmos essas matrizes estáticas no `ChromaDB`, prontas pra buscas na infraestrutura de *cosine similarity* (Similaridade Local do Cosseno).

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

print("🤖 Transformando texto nativo em Tensores Flutuantes e Populando o DB...")

# O 'text-embedding-3-small' da OpenAI fornece dimensões massivas nativamente baratas
motor_de_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Inserção no banco em memória
vector_db = Chroma.from_documents(documentos_fragmentados, motor_de_embeddings)

print("🎯 Espelho de vetores estabelecido com sucesso. Nosso cérebro consultivo inicial está alerta!")

---
## 🔍 Fase 3: RAG Core Pipeline (Transparência Sistêmica & Langfuse)

Nesta etapa o sistema ganha autonomia. Entretanto, injetaremos rastreabilidade corporativa imediata no cérebro orquestrador. 

O Framework Langfuse enxerga o mundo em instâncias decrescentes de tamanho:
1. **[TRACE]**: A jornada épica inteira. Desde o "Bom dia RAG" até o despejo final no chat na interface com o cliente.
2. **[SPAN]**: Uma jornada coadjuvante. Ex: "Vamos medir isoladamente em O-Notation se este Vector DB engasgou no I/O ao encontrar textos".
3. **[GENERATION]**: Um momento isolado de extrema atenção voltado especificamente a monitorar uma rede neural LLM gerando linguagem natural sob restrições (Custos em *Tokens*, Latência TTFT e Qualidade Sistêmica injetiva).

In [ ]:
from langfuse.decorators import observe, langfuse_context
from langfuse.callback import CallbackHandler
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# É ele que "escuta" eventos do LangChain e emite requisições silenciosas assincronamente ao dashboard cloud
handler_de_observabilidade = CallbackHandler()

### 3.1. Engrenagem Retriever Guiada (Span)

In [ ]:
# Configurando a ponte de busca: limitamos aos top K=2 documentos para manter o escopo enxuto
buscador = vector_db.as_retriever(search_kwargs={"k": 2})

# Observemos esta classe puramente computacional através de um Span
@observe(as_type="span", name="VectorDB_Retrieval_Event")
def buscar_materiais_do_banco(query: str):
    # O langchain resolve a matemática espacial da query contra nosso Chroma local 
    chunks_recuperados = buscador.invoke(query)
    textos_string = [doc.page_content for doc in chunks_recuperados]
    
    # Simulação Visual Acadêmica   
    print(f"\n🔎 [SISTEMA RETRIEVER] Procurando na biblioteca por: '{query}'")
    for i, texto in enumerate(textos_string, 1):
        print(f"  📄 Evidência {i}: {texto[:80]}...")
        
    return textos_string

### 3.2. Restrição Punitiva Neural (Generator)
Esta é a função mais perigosa de se projetar em RAG, pois o instinto natural dos LLMs generalistas é deduzir respostas preenchendo as lacunas e inventando mentiras aceitáveis morfologicamente (*Alucinações*). 
Neste escopo, nossa heurística é punitiva: "Responda se, e somente se, estiver neste exato papel virtual". 

In [ ]:
modelo_inteligente = ChatOpenAI(model="gpt-4o-mini", temperature=0.0) 

# O Prompt Template engessa o modelo dentro do contexto estritamente delimitado
prompt_restrito = ChatPromptTemplate.from_messages([
    ("system", "Você é um estrito analista textual restritivo e farmacêutico. Analise ESTRITAMENTE a aba 'EVIDÊNCIAS' para formular sua resposta à pergunta do paciente. Se a resposta demandada NÃO estiver nas EVIDÊNCIAS, você tem expressa orientação para declarar: 'Não possuo dados de suporte suficientes para responder.'\n\nEVIDÊNCIAS DISPONÍVEIS:\n{informacao_bruta}"),
    ("user", "Pergunta do Paciente: {duvida}")
])

@observe(as_type="generation", name="LLM_Formulator")
def formular_resposta(query: str, lista_de_evidencias: list):
    massa_contextual = "\n----\n".join(lista_de_evidencias)
    
    cadeia_rag = prompt_restrito | modelo_inteligente
    
    # Com os callbacks integrados do Langfuse no bloco invoke, nossa geração e consumo tokenizado vai direto ao relatório unificado.
    resposta = cadeia_rag.invoke(
        {"informacao_bruta": massa_contextual, "duvida": query},
        config={"callbacks": [handler_de_observabilidade]}
    )
    
    return resposta.content

### 3.3. Unificação Orquestrada (O Trace Absoluto)
Uma função soberana acionar os atores. É quem abre e finaliza a sessão RAG integral.

In [ ]:
@observe(as_type="trace", name="Sessao_Usuario_RAG")
def pipeline_rag_unificado(query: str):
    print("==================================================")
    print("🚀 INICIANDO PIPELINE RAG ACADÊMICO")
    print("==================================================")
    
    # Componente 1 / Fase Tática
    evidencias_recuperadas = buscar_materiais_do_banco(query)
    
    # Componente 2 / Fase Intelectual
    resposta_gerada = formular_resposta(query, evidencias_recuperadas)
    
    print(f"\n🧠 [SISTEMA GENERATOR] Transcrição elaborada ao paciente:\n=> {resposta_gerada}")
    print("==================================================")
    
    return resposta_gerada, evidencias_recuperadas

# ==========================================
# EXECUTANDO NOSSO TESTE END-TO-END
# ==========================================
pergunta_teste = "Causa problemas combinar ibuprofeno e aspirina na mesma janela diária?"
_ = pipeline_rag_unificado(pergunta_teste)

# Fechamos assincronia e despachamos obrigatoriamente relatórios suspensos via internet à nuvem do Langfuse
langfuse_context.flush()

## 🎉 Fechamento e Avaliação da Prática (Fases 1, 2 e 3)
Você consolidou a essência fundamental de um aplicativo base de Retrieval-Augmented Generation corporativo e injetou ferramentas pesadas de Telemetria e LLMOps com 30 linhas efetivas de implementação lógica.

**Seu próximo micro-passo antes da Fase 4 analítica, é validar seu Trace:**
1. Visite o **[Langfuse](https://cloud.langfuse.com/)** em sua Dashboard principal.
2. Navegue para o ambiente de **Traces**
3. Clique em **`Sessao_Usuario_RAG`** que acabou de aparecer na listagem.
4. Desdobre as hierarquias no diagrama estático:
   - Observe na lupa do `Generation` exatos quais Tokens enviamos e qual a String de Evidência maciça substituiu o espaço `informacao_bruta`.
   - Repare o quanto custou financeiramente ao seu centro de custo computacional esta pergunta isolada usando metadados pré-renderizados do *GPT-4o-Mini*.

> *Se você quiser simular uma Alucinação: Force via Input na query algo como 'Qual planeta o homem pousou?', observe a blindagem e em seguida a latência analítica do caso no trace*. 

Siga para a próxima etapa na nossa plataforma didática!